In [8]:
import matplotlib
# Force Matplotlib to use the standard windowing backend (Fix 2)
matplotlib.use('TkAgg')

import cv2
import json
import requests
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from pupil_apriltags import Detector
import os

# --- Configuration ---
JSON_FILENAME = "2026-rebuilt-welded.json"
JSON_URL = "https://raw.githubusercontent.com/wpilibsuite/allwpilib/main/apriltag/src/main/native/resources/edu/wpi/first/apriltag/2026-rebuilt-welded.json"
TAG_SIZE = 0.1651
CAMERA_PARAMS = [600, 600, 320, 240] # [fx, fy, cx, cy] - Calibrate this for better accuracy!

# --- Download JSON if missing ---
if not os.path.exists(JSON_FILENAME):
    print(f"Downloading {JSON_FILENAME}...")
    try:
        r = requests.get(JSON_URL)
        if r.status_code == 200:
            with open(JSON_FILENAME, 'wb') as f:
                f.write(r.content)
            print("✅ Download success.")
        else:
            print(f"❌ Error: Status {r.status_code}")
    except Exception as e:
        print(f"❌ Failed: {e}")
else:
    print(f"✅ Found {JSON_FILENAME} locally.")

✅ Found 2026-rebuilt-welded.json locally.


In [9]:
class FieldMapper:
    def __init__(self, json_path):
        self.tag_poses = {} 
        self.field_length = 17.55
        self.field_width = 8.05
        self.load_json(json_path)

    def load_json(self, path):
        with open(path, 'r') as f:
            data = json.load(f)
        
        if 'field' in data:
            self.field_length = data['field'].get('length', 17.55)
            self.field_width = data['field'].get('width', 8.05)

        for tag in data['tags']:
            id = tag['ID']
            tx = tag['pose']['translation']['x']
            ty = tag['pose']['translation']['y']
            tz = tag['pose']['translation']['z']
            qw = tag['pose']['rotation']['quaternion']['W']
            qx = tag['pose']['rotation']['quaternion']['X']
            qy = tag['pose']['rotation']['quaternion']['Y']
            qz = tag['pose']['rotation']['quaternion']['Z']
            self.tag_poses[id] = self.create_matrix(tx, ty, tz, qw, qx, qy, qz)
            
    def create_matrix(self, tx, ty, tz, qw, qx, qy, qz):
        xx, yy, zz = qx*qx, qy*qy, qz*qz
        xy, xz, yz = qx*qy, qx*qz, qy*qz
        wx, wy, wz = qw*qx, qw*qy, qw*qz

        R = np.array([
            [1 - 2*(yy+zz), 2*(xy-wz),   2*(xz+wy)],
            [2*(xy+wz),     1 - 2*(xx+zz), 2*(yz-wx)],
            [2*(xz-wy),     2*(yz+wx),     1 - 2*(xx+yy)]
        ])
        
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = [tx, ty, tz]
        return T

    def get_cam_position(self, tag_id, r_matrix, t_vec):
        if tag_id not in self.tag_poses:
            return None
        
        # 1. Get the Raw Optical Transform (Camera_Optical -> Tag_Optical)
        T_cam_opt_to_tag_opt = np.eye(4)
        T_cam_opt_to_tag_opt[:3, :3] = r_matrix
        T_cam_opt_to_tag_opt[:3, 3] = t_vec.flatten()
        
        # 2. Define the Basis Change Matrix (Optical -> NWU/WPILib)
        # Z (Forward) -> X
        # X (Right)   -> -Y
        # Y (Down)    -> -Z
        M_opt_to_nwu = np.array([
            [0, 0, 1, 0],
            [-1, 0, 0, 0],
            [0, -1, 0, 0],
            [0, 0, 0, 1]
        ])
        
        # 3. Convert the Detector Transform to NWU Frame (Basis Change)
        # T_cam_nwu_to_tag_nwu = M * T_raw * M_inv
        M_inv = np.linalg.inv(M_opt_to_nwu)
        T_cam_nwu_to_tag_nwu = M_opt_to_nwu @ T_cam_opt_to_tag_opt @ M_inv
        
        # 4. Invert to get (Tag_NWU -> Cam_NWU)
        T_tag_nwu_to_cam_nwu = np.linalg.inv(T_cam_nwu_to_tag_nwu)
        
        # 5. Chain with Field Map (Field -> Tag_NWU -> Cam_NWU)
        T_field_to_tag = self.tag_poses[tag_id]
        T_field_to_cam = T_field_to_tag @ T_tag_nwu_to_cam_nwu
        
        # 6. Extract Yaw
        yaw = np.arctan2(T_field_to_cam[1, 0], T_field_to_cam[0, 0])
        
        return T_field_to_cam[0, 3], T_field_to_cam[1, 3], T_field_to_cam[2, 3], yaw

# Re-Initialize mapper
mapper = FieldMapper(JSON_FILENAME)

In [12]:
# --- Setup Camera & Detector ---
cap = cv2.VideoCapture(0)
detector = Detector(families='tag36h11')

# --- Setup Matplotlib ---
plt.ion()
fig, ax = plt.subplots(figsize=(14, 8))

# 1. Draw Static Field Boundary
ax.plot([0, mapper.field_length, mapper.field_length, 0, 0], 
        [0, 0, mapper.field_width, mapper.field_width, 0], 'k-', linewidth=2)

# --- Plot Tags with Alternating Labels ---
tag_ids, tag_x, tag_y = [], [], []
tag_u, tag_v = [], [] 

for tid, pose in mapper.tag_poses.items():
    tag_ids.append(tid)
    tag_x.append(pose[0, 3])
    tag_y.append(pose[1, 3])
    
    # Extract Forward Vector for Arrows (Debugging)
    # Using Column 0 (X-axis) as Forward for this visual debug
    R = pose[:3, :3]
    normal_vector = R[:, 0] 
    tag_u.append(normal_vector[0])
    tag_v.append(normal_vector[1])

# Plot the Blue Squares (Tags)
ax.scatter(tag_x, tag_y, c='blue', marker='s', s=100, label='Tags', zorder=5)

# Plot the Orientation Arrows
ax.quiver(tag_x, tag_y, tag_u, tag_v, color='deepskyblue', scale=20, width=0.005, headwidth=3, zorder=6)

# --- ALTERNATING LABELS LOGIC ---
for i, tid in enumerate(tag_ids):
    if tid % 2 == 0:
        # Even IDs: Place ABOVE the tag
        offset = (0, 10)
        va = 'bottom'
    else:
        # Odd IDs: Place BELOW the tag
        offset = (0, -15)
        va = 'top'
        
    ax.annotate(str(tid), (tag_x[i], tag_y[i]), xytext=offset, 
                textcoords='offset points', ha='center', va=va,
                fontsize=9, fontweight='bold', color='navy')

# 2. Setup Robot Arrow
robot_arrow = ax.quiver(0, 0, 1, 0, color='red', scale=25, width=0.01, headwidth=4, zorder=10, label='Robot')

ax.set_title("Live Localization (Alternating Labels)")
ax.set_xlabel("Field X (m)")
ax.set_ylabel("Field Y (m)")
ax.axis('equal')
ax.grid(True)
ax.legend(loc='upper right')

print("Starting Loop... Press 'q' in CV2 window to stop.")

try:
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        # ... (Same vision processing loop as before) ...
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        tags = detector.detect(gray, estimate_tag_pose=True, 
                               camera_params=CAMERA_PARAMS, tag_size=TAG_SIZE)
        
        frame_x_estimates = []
        frame_y_estimates = []
        frame_yaw_vectors = [] 
        
        for tag in tags:
            corners = tag.corners.astype(int)
            for i in range(4):
                cv2.line(frame, tuple(corners[i]), tuple(corners[(i+1)%4]), (0, 255, 0), 2)
            
            # Note: Ensure you are using the FIXED FieldMapper class from previous step
            res = mapper.get_cam_position(tag.tag_id, tag.pose_R, tag.pose_t)
            
            if res:
                rx, ry, _, ryaw = res
                frame_x_estimates.append(rx)
                frame_y_estimates.append(ry)
                frame_yaw_vectors.append((np.cos(ryaw), np.sin(ryaw)))

                cv2.putText(frame, f"ID:{tag.tag_id}", (corners[0][0], corners[0][1]-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

        if len(frame_x_estimates) > 0:
            final_x = sum(frame_x_estimates) / len(frame_x_estimates)
            final_y = sum(frame_y_estimates) / len(frame_y_estimates)
            sum_cos = sum(v[0] for v in frame_yaw_vectors)
            sum_sin = sum(v[1] for v in frame_yaw_vectors)
            final_yaw = np.arctan2(sum_sin, sum_cos)
            
            u = np.cos(final_yaw)
            v = np.sin(final_yaw)
            robot_arrow.set_offsets([final_x, final_y])
            robot_arrow.set_UVC(u, v)
            plt.pause(0.01)

        cv2.imshow('Robot Camera', frame)
        if cv2.waitKey(1) == ord('q'):
            break

except KeyboardInterrupt:
    print("Stopped.")
finally:
    cap.release()
    cv2.destroyAllWindows()
    plt.close('all')

Starting Loop... Press 'q' in CV2 window to stop.
